In [4]:
%%writefile assignment1.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 10000000

__global__ void complementDNA(const char *input, char *output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        char c = input[idx];

        if (c == 'A') output[idx] = 'T';
        else if (c == 'T') output[idx] = 'A';
        else if (c == 'C') output[idx] = 'G';
        else if (c == 'G') output[idx] = 'C';
        else output[idx] = 'N';
    }
}

void generateDNA(char *dna, int n) {
    char bases[4] = {'A', 'C', 'G', 'T'};
    int i;

    for (i = 0; i < n; i++) {
        dna[i] = bases[rand() % 4];
    }
}

void printFirst50(char *arr) {
    int i;
    for (i = 0; i < 50; i++) {
        printf("%c ", arr[i]);
    }
    printf("\n");
}

void runExperiment(int blockSize) {
    char *h_input;
    char *h_output;
    char *d_input;
    char *d_output;

    int gridSize;
    long long totalThreads;

    cudaEvent_t start, stop;
    float ms = 0.0f;

    h_input = (char *)malloc(N * sizeof(char));
    h_output = (char *)malloc(N * sizeof(char));

    generateDNA(h_input, N);

    cudaMalloc((void **)&d_input, N * sizeof(char));
    cudaMalloc((void **)&d_output, N * sizeof(char));

    cudaMemcpy(d_input, h_input, N * sizeof(char), cudaMemcpyHostToDevice);

    gridSize = (N + blockSize - 1) / blockSize;
    totalThreads = (long long)gridSize * blockSize;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    complementDNA<<<gridSize, blockSize>>>(d_input, d_output, N);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_output, d_output, N * sizeof(char), cudaMemcpyDeviceToHost);

    printf("========================================\n");
    printf("Block size: %d\n", blockSize);
    printf("First 50 elements of original sequence:\n");
    printFirst50(h_input);
    printf("First 50 elements of complemented sequence:\n");
    printFirst50(h_output);
    printf("Total number of threads launched: %lld\n", totalThreads);
    printf("Kernel execution time: %.3f ms\n", ms);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);
    free(h_output);
}

int main() {
    int blockSizes[4] = {64, 128, 256, 512};
    int i;

    srand(time(NULL));

    for (i = 0; i < 4; i++) {
        runExperiment(blockSizes[i]);
    }

    return 0;
}

Writing assignment1.cu


In [5]:
!nvcc assignment1.cu -o a1
!./a1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Block size: 64
First 50 elements of original sequence:
C T C T G T T C T C G G T A G C A G T A A A C A G C C A T C G A C T T T T G A G T G A G T G A T A T 
First 50 elements of complemented sequence:
G A G A C A A G A G C C A T C G T C A T T T G T C G G T A G C T G A A A A C T C A C T C A C T A T A 
Total number of threads launched: 10000000
Kernel execution time: 0.491 ms
Block size: 128
First 50 elements of original sequence:
T C T G T A T A T C T A T G A G A C T A C T C G T T C T C T C C C A T A C T C A A A A T G C C T G A 
First 50 elements of complemented sequence:
A G A C A T A T A G A T A C T C T G A T G A G C A A G A G A G G G T A T G A G T T T T A C G G A C T 
Total number of threads launched: 10000000
Kernel execution time: 0.283 ms
Block size: 256
First 50 elements of original sequence:
T G 